# 02 · Esha · Run all three planners + evaluation
Run the cells **top to bottom**. If Colab restarted, start again from cell 1.

Needs: Colab Secret `GEMINI_API_KEY` (key icon on the left, Notebook access ON).

## 1. Get the latest code

In [ ]:
import os
if not os.path.exists('/content/daybeam'):
    !git clone -q https://github.com/equivalentpay0910/daybeam.git /content/daybeam
%cd /content/daybeam
!git pull -q
!pip install -q -r requirements.txt

## 2. API key + model (the key is read from Colab Secrets, never typed here)

In [ ]:
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['DAYBEAM_MODEL'] = 'gemini-3.8-flash'

## 3. Tests (fake Gemini, free). Should say `6 passed`

In [ ]:
!pytest -q

## 4. Try each planner on one dev task (real Gemini)

In [ ]:
import json
from core.schemas import Task
from agent.planner import plan
from agent.toolbox import check_plan, get_place, which_tools

task = Task(**json.load(open('data/tasks_dev/D001.json')))
print(task.request_text, '\n')
for method in ['one_shot', 'react', 'beam']:
    r = plan(task, method=method)
    print(f'===== {method} =====')
    if r.error:
        print('ERROR:', r.error); continue
    c = check_plan(task, r.plan)
    for t in c.timeline:
        print(f"  {t['arrive']}-{t['depart']}  {t['stop_id']}  {get_place(t['place_id']).name}")
    print(f"  home {c.end_time} | drive {c.total_drive_min} min | ${c.total_cost_usd:g}")
    print('  PASS' if c.passed else '  FAIL: ' + '; '.join(v.detail for v in c.violations))
    print(f'  LLM calls {r.llm_calls} | tokens {r.tokens_in + r.tokens_out} | {r.seconds:.1f}s\n')
print('Tools used:', which_tools())

## 5. Quick evaluation on the 3 dev tasks (3 planners × 3 tasks × 1 seed)
Switch `--tasks data/tasks` once Kami's tasks are in the repo, and use `--seeds 0,1,2` for the real run.

In [ ]:
!python -m eval.run_eval --tasks data/tasks_dev --seeds 0 --out eval/results/dev_results.csv

## 6. Summary table + charts

In [ ]:
!python -m eval.analyze eval/results/dev_results.csv
from IPython.display import Image, display
for f in ['chart_success', 'chart_tradeoff', 'chart_difficulty']:
    display(Image(f'eval/results/{f}.png', width=520))

## 7. Save charts to your Google Drive (for the slides)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/CS 156 AI Project/charts'
!cp eval/results/*.png eval/results/*.csv '/content/drive/MyDrive/CS 156 AI Project/charts/'
print('Copied to Drive > CS 156 AI Project > charts')